# DocLens with Qwen3-VL-8B on Kaggle: reproducing Table 9 (MMLongBench-Doc)

The DocLens paper (Appendix C, Table 9) evaluates the open-source **Qwen3-VL-8B-Instruct** on a random sample of
**200 MMLongBench-Doc questions**:

| Method | Paper |
|---|---|
| Vanilla Qwen3-VL-8B (screenshots only) | 43.0 |
| Qwen3-VL-8B + OCR | 42.9 |
| **DocLens w/ Qwen3-VL-8B** | **61.1** |

This notebook re-runs those three rows with the official DocLens code and the paper's settings (Te = Ta = 8,
temperature 0.7, the paper's Adjudicator prompt). Qwen runs locally on Kaggle's **2x T4** through **vLLM**, so there are no
API limits and no cost.

**Before running**
* *Settings → Accelerator*: **GPU T4 x2** · *Internet*: **On**.
* *Add Input* (right panel) → *Your Work* → your earlier notebook → the version that contains **`parsed_MMLongBenchDoc.tar`**
  (the saved MinerU results). Then **Run All**.
* GPU budget: Kaggle gives 30 GPU hours per week and 12 per session. Every run resumes from its checkpoint file, so if a
  session ends, open a new one, re-attach the input and **Run All** again.

**Differences from the paper, forced by the T4 GPUs** (state them in your paper):
* The context is limited to the T4s' memory, so the Page Navigator reads long documents in chunks of `NAV_CHUNK` pages
  (the paper does the same with 50-page chunks for Claude, Appendix D.1). Prompts that still do not fit get their images
  halved and are retried (also Appendix D.1).
* Answer extraction for scoring runs on the same Qwen model instead of Gemini-2.5-Flash.
* The paper's 200 questions are not published; this notebook draws its own seeded random 200.

In [ ]:
# ======================= SETTINGS =======================
WORK = "/kaggle/working"
CDC_REPO = "https://github.com/Mariam1543/aqqal123.git"
CDC_BRANCH = "claude/wizardly-rubin-kdthhy"
DATASET = "MMLongBenchDoc"

N_QUESTIONS = 200                 # Table 9 uses a random sample of 200 questions
SEED = 0
RUN_ONLY = ["qwen_doclens", "qwen_ocr", "qwen_vanilla"]   # in this order: the main result first

MAX_MODEL_LEN = 40960             # context that fits next to the 8B weights on 2x T4
NAV_CHUNK = 20                    # Page Navigator pages per request
MAX_IMAGES = 48                   # images per request accepted by the server
MAX_CONCURRENT = 4
PAPER_PROMPTS = True
PILOT_QUESTIONS = 5

## 1. Get the code

In [ ]:
import os, sys, subprocess, shutil, json, glob, random, time
os.chdir(WORK)
if not os.path.exists("DocLens"):
    subprocess.run(["git", "clone", "https://github.com/dwzhu-pku/DocLens.git"], check=True)
shutil.rmtree("doclens_cdc", ignore_errors=True)   # always take the latest version of the extension
shutil.rmtree("_cdc_repo", ignore_errors=True)
subprocess.run(["git", "clone", "--depth", "1", "-b", CDC_BRANCH, CDC_REPO, "_cdc_repo"], check=True)
shutil.copytree("_cdc_repo/doclens_cdc", "doclens_cdc")
shutil.rmtree("_cdc_repo")
DOCLENS, CDC = f"{WORK}/DocLens", f"{WORK}/doclens_cdc"
ROOT = f"{WORK}/full_{DATASET}"
RESULTS = f"{WORK}/results_qwen/{DATASET}"
os.makedirs(RESULTS, exist_ok=True)
sys.path.insert(0, CDC)
from reproduction.experiments import EXPERIMENTS, run_args
SELECTED = {k: EXPERIMENTS[k] for k in RUN_ONLY}
for k, e in SELECTED.items():
    print(f"{k:14s} {e['note']:40s} K={e['k1']}/{e['k2']}")

## 2. Install vLLM and the DocLens dependencies (about 10 minutes)

In [ ]:
!pip install -q "vllm>=0.11.0" 2>&1 | tail -n 3
!pip install -q -r {CDC}/requirements-kaggle.txt 2>&1 | tail -n 2
!python -c "import vllm, torch; print('vllm', vllm.__version__, '| torch', torch.__version__, '| GPUs', torch.cuda.device_count())"

## 3. Restore the MinerU results and re-create the page images

In [ ]:
tars = glob.glob("/kaggle/input/**/parsed_MMLongBenchDoc.tar", recursive=True)
assert tars, "Add the saved notebook version containing parsed_MMLongBenchDoc.tar as an input (right panel: Add Input)."
if not os.path.exists(f"{ROOT}/clean/data/{DATASET}/documents"):
    !tar -xf {tars[0]} -C {WORK}
!python {CDC}/scripts/download_data.py --dataset {DATASET} --out {DOCLENS}/data
SAMPLES = f"{DOCLENS}/data/{DATASET}/samples.json"
DOCUMENTS = f"{DOCLENS}/data/{DATASET}/documents"
# The tar holds MinerU's output but not the page screenshots; re-render them (skips existing ones).
!python {CDC}/scripts/prepare_subset.py --all --samples {SAMPLES} --documents {DOCUMENTS} --out_root {ROOT} --dataset_name {DATASET} --workers 4
docs = f"{ROOT}/clean/data/{DATASET}/documents"
print(len(glob.glob(f"{docs}/*/*.jpeg")), "page images,", len(glob.glob(f"{docs}/*/MinerU_Page*/*.md")), "parsed pages (expected 6529 each)")

In [ ]:
# A seeded random sample of N_QUESTIONS questions, kept in the benchmark's order
all_q = json.load(open(SAMPLES))
picked = set(random.Random(SEED).sample(range(len(all_q)), N_QUESTIONS))
QUESTIONS = f"{ROOT}/samples_random{N_QUESTIONS}_seed{SEED}.json"
json.dump([q for i, q in enumerate(all_q) if i in picked], open(QUESTIONS, "w"), ensure_ascii=False, indent=1)
print(N_QUESTIONS, "questions from", len({q['doc_id'] for i, q in enumerate(all_q) if i in picked}), "documents ->", QUESTIONS)

## 4. Start the Qwen3-VL-8B server (downloads ~17 GB the first time, then loads onto both T4s)

In [ ]:
import urllib.request
log = open(f"{WORK}/vllm.log", "w")
server = subprocess.Popen([
    "vllm", "serve", "Qwen/Qwen3-VL-8B-Instruct",
    "--tensor-parallel-size", "2", "--dtype", "half",
    "--max-model-len", str(MAX_MODEL_LEN), "--gpu-memory-utilization", "0.92",
    "--limit-mm-per-prompt", json.dumps({"image": MAX_IMAGES}),
    "--mm-processor-kwargs", json.dumps({"max_pixels": 1003520}),
    "--max-num-seqs", "16", "--enable-prefix-caching", "--enforce-eager",
    "--disable-custom-all-reduce", "--port", "8000",
], stdout=log, stderr=subprocess.STDOUT)

start = time.time()
while True:
    try:
        urllib.request.urlopen("http://localhost:8000/v1/models", timeout=5)
        print(f"Server ready after {(time.time() - start) / 60:.1f} min"); break
    except Exception:
        if server.poll() is not None or time.time() - start > 40 * 60:
            print(open(f"{WORK}/vllm.log").read()[-6000:])
            raise RuntimeError("vLLM did not start: send the log above.")
        time.sleep(20)

In [ ]:
# Smoke test: one real question about one real page
import base64, openai
client = openai.OpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY")
page = sorted(glob.glob(f"{docs}/*/1.jpeg"))[0]
b64 = base64.b64encode(open(page, "rb").read()).decode()
r = client.chat.completions.create(model="Qwen/Qwen3-VL-8B-Instruct", max_tokens=60, messages=[{"role": "user", "content": [
    {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}},
    {"type": "text", "text": "What is the title of this page?"}]}])
print(page, "->", r.choices[0].message.content)

## 5. Pilot: 5 questions, to check everything and measure the time per question

In [ ]:
common = (f"python {CDC}/run_cdc.py --doclens_root {DOCLENS} --data_root {ROOT}/clean --variant clean "
          f"--dataset_name {DATASET} --samples {QUESTIONS} --max_concurrent {MAX_CONCURRENT} "
          f"--qwen_base_url http://localhost:8000/v1 --qwen_max_images {MAX_IMAGES} --nav_chunk_pages {NAV_CHUNK} "
          f"{'--paper_prompts' if PAPER_PROMPTS else ''}")
t0 = time.time()
args = run_args(SELECTED["qwen_doclens"])
!{common} {args} --limit {PILOT_QUESTIONS} --output {RESULTS}/pilot_qwen_doclens.jsonl
per_q = (time.time() - t0) / PILOT_QUESTIONS
rows = [json.loads(l) for l in open(f"{RESULTS}/pilot_qwen_doclens.jsonl")]
ok = [r for r in rows if "final_score" in r]
print(f"{len(ok)}/{len(rows)} scored; ~{per_q / 60:.1f} min per question (with {MAX_CONCURRENT} in parallel: less)")
print(f"Rough estimate for {N_QUESTIONS} questions of DocLens: {per_q * N_QUESTIONS / 3600:.1f} h (upper bound)")
for r in rows:
    if "cdc_error" in r:
        print("ERROR:", r["cdc_error"][:600]); break

## 6. The Table 9 runs
Each run writes `results_qwen/MMLongBenchDoc/<experiment>.jsonl` and **resumes** after an interruption.
`qwen_doclens` (the main result) runs first. If the session ends, start a new one, re-attach the input and **Run All**.

In [ ]:
for key, exp in SELECTED.items():
    print("=" * 30, key, "-", exp["note"])
    args = run_args(exp)
    !{common} {args} --output {RESULTS}/{key}.jsonl

## 7. Compare with the paper

In [ ]:
!python {CDC}/scripts/report_reproduction.py --results_dir {RESULTS} --dataset_name {DATASET} \
    --expected_questions {N_QUESTIONS} --out {RESULTS}/reproduction_report.md
from IPython.display import Markdown, display
display(Markdown(open(f"{RESULTS}/reproduction_report.md").read()))

## 8. Save the results
*Save Version → Quick Save → Advanced Settings → "Save output for this version"*. Only `results_qwen/` is small enough to matter;
the MinerU data is already safe in the earlier version.

Before saving, remove the large folders (they are restored from the input next time), so the upload stays small:

In [ ]:
CLEANUP_BEFORE_SAVE = False   # set True and run this cell only when you are about to save and are done for now
if CLEANUP_BEFORE_SAVE:
    !rm -rf {ROOT} {DOCLENS}/data
!du -sh {WORK}